In [2]:
# =====================================================================
# LAB 1: NLU BANCÁRIO COM REGRESSÃO LOGÍSTICA
# =====================================================================

import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression


dados_treino = [
    # CONSULTAR SALDO
    ("quero consultar meu saldo", "consultar_saldo"),
    ("qual é o meu saldo", "consultar_saldo"),
    ("quanto dinheiro tenho na conta", "consultar_saldo"),
    ("quero saber meu saldo", "consultar_saldo"),
    ("mostre o saldo da minha conta", "consultar_saldo"),
    ("quanto tenho disponível", "consultar_saldo"),

    # TRANSFERÊNCIA
    ("quero transferir 100 reais", "transferencia"),
    ("transferir 250 reais", "transferencia"),
    ("quero fazer uma transferência", "transferencia"),
    ("preciso transferir dinheiro", "transferencia"),
    ("mande 500 reais para outra conta", "transferencia"),
    ("quero enviar 80 reais", "transferencia"),

    # PAGAMENTO
    ("quero pagar 100 reais", "pagamento"),
    ("preciso pagar uma conta", "pagamento"),
    ("quero pagar meu boleto", "pagamento"),
    ("pagar 250 reais do boleto", "pagamento"),
    ("quero realizar um pagamento", "pagamento"),
    ("preciso pagar 300 reais", "pagamento"),

    # SAQUE
    ("quero sacar 100 reais", "saque"),
    ("preciso sacar dinheiro", "saque"),
    ("quero fazer um saque", "saque"),
    ("sacar 200 reais", "saque"),
    ("quero retirar 50 reais", "saque"),
    ("preciso retirar dinheiro da conta", "saque"),

    # FORA DO ESCOPO
    ("qual é a previsão do tempo", "fora_escopo"),
    ("me conte uma piada", "fora_escopo"),
    ("qual é a capital do brasil", "fora_escopo"),
    ("quem ganhou o jogo", "fora_escopo"),
    ("qual é a temperatura hoje", "fora_escopo"),
    ("me fale sobre futebol", "fora_escopo")
]


stopwords = [
    "a", "o", "as", "os", "um", "uma", "de", "da", "do",
    "das", "dos", "para", "por", "me", "meu", "minha",
    "na", "no", "nas", "nos", "e", "é"
]


# =====================================================================
# 1. PRÉ-PROCESSAMENTO
# =====================================================================

def pré_processar(texto):

    texto_limpo = re.sub(
        r'[^\w\s]',
        '',
        texto.lower()
    ).strip()

    tokens = texto_limpo.split()

    tokens_filtrados = [
        palavra
        for palavra in tokens
        if palavra not in stopwords
    ]

    return " ".join(tokens_filtrados)


X_treino_limpo = [
    pré_processar(item[0])
    for item in dados_treino
]

y_treino = [
    item[1]
    for item in dados_treino
]


# =====================================================================
# 2. VETORIZAÇÃO E TREINAMENTO
# =====================================================================

vectorizer = TfidfVectorizer()


X_vetorizado = vectorizer.fit_transform(
    X_treino_limpo
)


modelo = LogisticRegression(
    max_iter=1000
)


modelo.fit(
    X_vetorizado,
    y_treino
)


# =====================================================================
# 3. EXTRAÇÃO DE ENTIDADE (REGEX)
# =====================================================================

def extrair_valor_reais(texto_original):

    match = re.search(
        r'(\d+)\s*reais',
        texto_original,
        re.IGNORECASE
    )

    if match:
        valor = float(
            match.group(1)
        )

        return valor

    return None


# =====================================================================
# 4. PIPELINE NLU
# =====================================================================

def processar_nlu(mensagem_usuario, threshold=0.55):

    texto_p = pré_processar(
        mensagem_usuario
    )

    vetor_input = vectorizer.transform(
        [texto_p]
    )

    probas = modelo.predict_proba(
        vetor_input
    )[0]


    # Maior probabilidade
    maior_confianca = max(probas)


    # Posição da maior probabilidade
    indice_maior = probas.argmax()


    # Classe correspondente
    intencao_prevista = modelo.classes_[
        indice_maior
    ]


    # Extrair valor
    valor = extrair_valor_reais(
        mensagem_usuario
    )


    # Verificar threshold
    if (
        maior_confianca < threshold
        or
        intencao_prevista == "fora_escopo"
    ):

        return {
            "status": "FALLBACK",
            "intencao": intencao_prevista,
            "confianca": maior_confianca,
            "valor": valor
        }

    else:

        return {
            "status": "SUCESSO",
            "intencao": intencao_prevista,
            "confianca": maior_confianca,
            "valor": valor
        }


# =====================================================================
# TESTE
# =====================================================================

if __name__ == "__main__":

    mensagem = "quero transferir 100 reais"

    resultado = processar_nlu(
        mensagem
    )

    print(resultado)

{'status': 'FALLBACK', 'intencao': np.str_('transferencia'), 'confianca': np.float64(0.4308700837040201), 'valor': 100.0}
